# XGBoost & Optuna — Hyperparameter Tuning

In [ ]:
# Day 29 — XGBoost & Optuna
# XGBoost = the most popular ML algorithm for STRUCTURED/TABULAR data
# Optuna = a smart library for finding the BEST hyperparameters
#
# In Day 28, we trained RandomForest with default settings
# But HOW do we pick the best settings (n_estimators, max_depth, etc.)?
# That's what hyperparameter tuning does.
#
# Today we'll cover:
#   1. What is XGBoost and why it wins Kaggle competitions
#   2. Training an XGBoost model
#   3. What are hyperparameters vs parameters
#   4. GridSearchCV — try all combinations (brute force)
#   5. Optuna — smart search (finds good params faster)
#   6. Putting it all together: XGBoost + Optuna
#
# Install: pip install xgboost optuna

### XGBoost — eXtreme Gradient Boosting

In [ ]:
# XGBoost is a BOOSTED tree algorithm
#
# Random Forest: many trees vote INDEPENDENTLY (parallel)
# XGBoost: trees are built SEQUENTIALLY — each new tree
#          focuses on the MISTAKES of the previous trees
#
# Analogy:
# Random Forest = 100 students take the same exam independently
# XGBoost = student 1 takes exam, gets some wrong,
#           student 2 focuses on those wrong answers,
#           student 3 focuses on what student 2 still got wrong...
#           Each student CORRECTS the mistakes of the previous one

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Create the same server health dataset
np.random.seed(42)
n = 500
cpu = np.random.randint(10, 100, n)
memory = np.random.randint(15, 95, n)
disk = np.random.randint(10, 90, n)
error_count = np.random.randint(0, 50, n)
score = 0.3 * cpu + 0.25 * memory + 0.15 * disk + 0.8 * error_count + np.random.normal(0, 10, n)
label = (score > 50).astype(int)

X = pd.DataFrame({'cpu': cpu, 'memory': memory, 'disk': disk, 'error_count': error_count})
y = label
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Training XGBoost — same sklearn API!

from xgboost import XGBClassifier

# XGBoost follows the same fit/predict pattern as sklearn
xgb = XGBClassifier(
    n_estimators=100,        # number of trees
    max_depth=5,             # how deep each tree can grow
    learning_rate=0.1,       # how much each tree contributes
    random_state=42,
    eval_metric='logloss'    # suppress warnings
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print(f'XGBoost Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, target_names=['healthy', 'critical']))

In [ ]:
# Feature importance from XGBoost

import matplotlib.pyplot as plt

importances = xgb.feature_importances_
features = X.columns.tolist()
sorted_idx = np.argsort(importances)

plt.figure(figsize=(8, 4))
plt.barh(range(len(sorted_idx)), importances[sorted_idx], color='#e74c3c')
plt.yticks(range(len(sorted_idx)), [features[i] for i in sorted_idx])
plt.title('Feature Importance (XGBoost)', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

### Hyperparameters vs Parameters

In [ ]:
# PARAMETERS = what the model LEARNS from data
#   Example: weights in linear regression, split rules in trees
#   You DON'T set these — the model learns them during training
#
# HYPERPARAMETERS = settings you set BEFORE training
#   Example: n_estimators, max_depth, learning_rate
#   You CHOOSE these — they control HOW the model learns
#
# Analogy:
# Parameters = what a student learns (knowledge)
# Hyperparameters = how the student studies (study method, hours per day, etc.)
#
# Key XGBoost hyperparameters:
# n_estimators  -> number of trees (more = potentially better but slower)
# max_depth     -> depth of each tree (deeper = more complex, risk overfitting)
# learning_rate -> step size (smaller = more careful learning, needs more trees)
# subsample     -> fraction of data used per tree (0.5-1.0)
# colsample_bytree -> fraction of features used per tree (0.5-1.0)
# min_child_weight -> minimum samples in a leaf (prevents overfitting)

print('Finding the best combination of these settings = HYPERPARAMETER TUNING')

### GridSearchCV — Brute Force Search

In [ ]:
# GridSearchCV tries EVERY combination of hyperparameters
# and picks the one with the best cross-validation score

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# Define the grid of hyperparameters to try
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}
# Total combinations: 3 × 3 × 3 = 27
# With 5-fold CV: 27 × 5 = 135 model trainings!

grid = GridSearchCV(
    estimator=XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    cv=5,                    # 5-fold cross-validation
    scoring='accuracy',
    verbose=1                # show progress
)

grid.fit(X_train, y_train)

print(f'\nBest parameters: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.2%}')
print(f'Test accuracy: {grid.score(X_test, y_test):.2%}')

# Problem: GridSearch is SLOW for large grids
# If you have 10 hyperparameters × 5 values each = 5^10 = ~10 million combos!
# This is where Optuna comes in...

### Optuna — Smart Hyperparameter Search

In [ ]:
# Optuna uses a SMART search strategy (Bayesian optimization)
# Instead of trying every combination, it:
# 1. Tries a few random combinations
# 2. Learns WHICH regions of the search space are promising
# 3. Focuses on those promising regions
# → Finds great hyperparameters in MUCH fewer trials

import optuna
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier

# Suppress Optuna's info logs for cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """Optuna calls this function for each trial.
    It suggests hyperparameters, trains a model, and returns the score."""

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'eval_metric': 'logloss',
        'random_state': 42
    }

    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

# Run the search
study = optuna.create_study(direction='maximize')    # maximize accuracy
study.optimize(objective, n_trials=50)               # try 50 combinations

print(f'Best accuracy: {study.best_value:.2%}')
print(f'Best params: {study.best_params}')

In [ ]:
# Train the final model with the best parameters

best_params = study.best_params
best_params['eval_metric'] = 'logloss'
best_params['random_state'] = 42

best_model = XGBClassifier(**best_params)
best_model.fit(X_train, y_train)

test_accuracy = best_model.score(X_test, y_test)
print(f'Final Test Accuracy: {test_accuracy:.2%}')

print(classification_report(y_test, best_model.predict(X_test),
                           target_names=['healthy', 'critical']))

In [ ]:
# Visualise Optuna's search

import matplotlib.pyplot as plt

# Plot optimization history
trials = [t.value for t in study.trials]

plt.figure(figsize=(10, 4))
plt.plot(range(len(trials)), trials, 'o-', alpha=0.5, markersize=4, color='#3498db')
plt.axhline(y=study.best_value, color='red', linestyle='--',
            label=f'Best: {study.best_value:.4f}')
plt.title('Optuna Optimisation History', fontweight='bold')
plt.xlabel('Trial Number')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Practice Questions:

# 1. Compare XGBoost (default) vs RandomForest (default) on the same data
#    Print accuracy, precision, recall, F1 for both

# 2. Use GridSearchCV with XGBoost to find the best:
#    n_estimators: [100, 200], max_depth: [3, 5, 7], learning_rate: [0.05, 0.1]

# 3. Use Optuna to tune a RandomForestClassifier instead of XGBoost
#    Tune: n_estimators, max_depth, min_samples_split, min_samples_leaf

# 4. Create a function tune_model(X, y, n_trials=30) that:
#    Uses Optuna to tune XGBoost and returns the best model and params

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: Production-Grade Model Selection
# Using the server health data, compare 4 models:
#   LogisticRegression, DecisionTree, RandomForest, XGBoost
# For each: use 5-fold CV, report mean accuracy ± std
# Create a bar chart comparing all 4 models
# Pick the best one and tune it with Optuna (30 trials)

# Exercise 2: Automated ML Pipeline
# Write a function auto_tune(X, y, model_type='xgboost'):
#   - Splits data 80/20
#   - Uses Optuna to find best hyperparameters (20 trials)
#   - Trains the final model with best params
#   - Returns: best_model, best_params, test_accuracy
# Support both 'xgboost' and 'random_forest' model types

# Exercise 3: MLOps — Model Registry Preparation
# After tuning, create a dict with everything needed for deployment:
# model_info = {
#   'model_name': 'server_health_classifier',
#   'model_type': 'XGBClassifier',
#   'best_params': {...},
#   'accuracy': 0.95,
#   'f1': 0.93,
#   'features': ['cpu', 'memory', 'disk', 'error_count'],
#   'tuning_method': 'optuna',
#   'n_trials': 50
# }
# Save it as JSON — this is what gets logged to MLflow (Day 34)!